>Feature engineering transforms raw transaction data into behavioral signals that help reveal abnormal patterns. In anomaly detection, the quality of features often matters more than the choice of model.
>Effective fraud detection focuses on behavioral rather than static features, with time playing a central role in capturing changes in activity. At the same time, care must be taken to avoid information leakage, as it can lead to misleading model performance.

# PaySim

## Time Handling & Ordering

Rolling time-based features are designed to capture short-term behavioral patterns rather than static transaction attributes. In fraud detection, transactions are often only suspicious when viewed in the context of recent user activity.

To avoid data leakage, all rolling features are computed using only past transactions, so each feature reflects information available at the time of the transaction.

The choice of rolling window is heuristic and domain-driven. In PaySim, a window of 24 time steps is used as an approximation of one day, enabling the model to capture daily activity patterns while remaining computationally efficient.

In [1]:
import pandas as pd

paysim = pd.read_csv("../data/raw/paysim.csv")

In [2]:
# sort step feature
paysim = paysim.sort_values(["nameOrig", "step"]).reset_index(drop=True)

In [4]:
# transaction count (24h)
paysim["tx_count_last_24h"] = (
    paysim.groupby("nameOrig")["amount"] # counting rows (any random column would do)
    .rolling(window=24, min_periods=1)
    .count()
    .reset_index(level=0, drop=True)
)

In [7]:
paysim.groupby("nameOrig")["step"].is_monotonic_increasing.all()

np.True_

This feature captures transaction velocity by counting how many transactions each sender performs within a 24-hour rolling window, which helps identify abnormal bursts of activity commonly associated with fraud.

In [ ]:
# average transaction amount (24h)
paysim["avg_amount_last_24h"] = (
    paysim.groupby("nameOrig")["amount"]
    .rolling(window=24, min_periods=1)
    .mean()
    .reset_index(level=0, drop=True)
)

This feature captures the average amount of transactions for each sender in the recent 24 hour window.

In [ ]:
# sum of amount (24h)
paysim["sum_amount_last_24h"] = (
    paysim.groupby("nameOrig")["amount"]
      .rolling(window=24, min_periods=1)
      .sum()
      .reset_index(level=0, drop=True)
)